In [19]:
import pandas as pd

df = pd.read_csv("data_clean.csv")
df_original = pd.read_csv("data.csv")

In [21]:
df.columns

Index(['Age ', 'BMI', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Last BMI on file', 'Length of followup',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Intraventricular extension', 'CSF leak', 'Mortality Y/N',
       'Prior operation', 'recurring tumor?',
       'Recurrence after this operation', 'Shunt (y/n) ', 'Transient DI',
       'Permanent DI',
       'DDAVP (y/n) - most patients get periop, can do time frame (in hospital vs 1 mo vs 1 yr postop)',
       'DDAVP outside of hospital', 'DDAVP ≥1 year', 'Post-op new panhypopit',
       'Other vascular complication (need to specify)',
       'Hemorrhage or hemorrhagic stroke', 'Ischemic Stroke',
       'Other intracranial infection', 'Non-intracranial infectio

In [33]:
import pandas as pd
from pycaret.classification import setup, compare_models, pull, evaluate_model


# --- Example: predict CSF leak ---
target = "Transient DI"

# Feature space: pre-op + intra-op (you can adjust which to include)
features = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Solid, cystic, mixed_cystic", "Solid, cystic, mixed_mixed",
    "Solid, cystic, mixed_solid", "Prior operation", "Surgical Approach_craniotomy", "Surgical Approach_endoscopic", 
    "Resection outcome (total, subtotal, partial)_near total",
    "Resection outcome (total, subtotal, partial)_subtotal",
    "Resection outcome (total, subtotal, partial)_total"
]

# Setup PyCaret AutoML environment
clf_setup = setup(
    data=df[features + [target]],
    target=target,
    session_id=42,
    fold=5,          # cross-validation
    use_gpu=False,   # dataset is small, CPU is fine
    verbose=False
)

# Compare models
best_model = compare_models(sort="AUC")  # or "Accuracy" for binary
results = pull()  # get comparison table

print(results)

# Evaluate best model with detailed plots
evaluate_model(best_model)


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.7806,0.5405,0.4000,0.1667,0.2333,0.1684,0.1999,0.2200
qda,Quadratic Discriminant Analysis,0.1694,0.5357,1.0000,0.1694,0.2861,0.0000,0.0000,0.0140
nb,Naive Bayes,0.4167,0.5286,0.4000,0.0650,0.1111,-0.1026,-0.1127,0.0120
lightgbm,Light Gradient Boosting Machine,0.8306,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0500
dummy,Dummy Classifier,0.8306,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0160
dt,Decision Tree Classifier,0.6583,0.4810,0.2000,0.0500,0.0800,-0.0914,-0.0725,0.0160
ridge,Ridge Classifier,0.7833,0.4762,0.2000,0.1000,0.1333,0.0566,0.0646,0.0180
lda,Linear Discriminant Analysis,0.6333,0.4762,0.4000,0.0800,0.1333,-0.0002,0.0507,0.0140
gbc,Gradient Boosting Classifier,0.6833,0.4524,0.4000,0.1000,0.1600,0.0252,0.0697,0.0340
et,Extra Trees Classifier,0.7806,0.4464,0.2000,0.0500,0.0800,0.0500,0.0756,0.0460


                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.7806  0.5405     0.4  0.1667   
qda       Quadratic Discriminant Analysis    0.1694  0.5357     1.0  0.1694   
nb                            Naive Bayes    0.4167  0.5286     0.4  0.0650   
lightgbm  Light Gradient Boosting Machine    0.8306  0.5000     0.0  0.0000   
dummy                    Dummy Classifier    0.8306  0.5000     0.0  0.0000   
dt               Decision Tree Classifier    0.6583  0.4810     0.2  0.0500   
ridge                    Ridge Classifier    0.7833  0.4762     0.2  0.1000   
lda          Linear Discriminant Analysis    0.6333  0.4762     0.4  0.0800   
gbc          Gradient Boosting Classifier    0.6833  0.4524     0.4  0.1000   
et                 Extra Trees Classifier    0.7806  0.4464     0.2  0.0500   
lr                    Logistic Regression    0.7806  0.4452     0.0  0.0000   
catboost              CatBoost Classifier    0.7806 

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [35]:
import pandas as pd
from pycaret.classification import setup, compare_models, tune_model, pull
from sklearn.exceptions import NotFittedError

# Load your raw dataset
df_original = pd.read_csv("data.csv")

# ---- Feature groups (for convenience) ----
preop_basic = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension"
]

intraop_basic = [
    "Surgical Approach",                 # raw categorical in df_original
    "Solid, cystic, mixed",              # raw categorical in df_original
    "Resection outcome (total, subtotal, partial)"
]

location_histology = [
    "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)",
    "Tumor (Adamantinomatous, papillary)",
    "BRAF+"
]

# ---- Build 15 experiments (targets + feature sets) ----
experiments = [
    # Post-op complications
    {"target": "Permanent DI",
     "features": preop_basic + ["Surgical Approach", "Resection outcome (total, subtotal, partial)"]},




    # Composite endpoints (you can create these columns beforehand if desired)
    # Example creation idea (run before the loop):
    # df_original["Any complication"] = (
    #     (df_original["CSF leak"]==1) | (df_original["Transient DI"]==1) |
    #     (df_original["Permanent DI"]==1) | (df_original["Other vascular complication (need to specify)"]==1) |
    #     (df_original["Hemorrhage or hemorrhagic stroke"]==1) | (df_original["Ischemic Stroke"]==1) |
    #     (df_original["Other intracranial infection"]==1) | (df_original["Non-intracranial infection"]==1) |
    #     (df_original["Meningitis"]==1)
    # ).astype(int)

    # If you already created it, uncomment the experiment below:
    # {"target": "Any complication",
    #  "features": preop_basic + intraop_basic + location_histology},

    # Visual outcome (binary version you can precompute; multi-class needs classification setup too)
    # df_original["VisualImproved"] = (df_original[
    #   "Post-op visual improvement/stable/worse (subjective); should add something specific about visual fields if any patients have; add about visual acuity"
    # ].astype(str).str.lower().str.contains("improv")).astype(int)

    # {"target": "VisualImproved",
    #  "features": preop_basic + ["Surgical Approach"] + location_histology},

    # Two more variations to reach 15 now (without needing new columns):
    {"target": "Other intracranial infection",
     "features": preop_basic + ["Surgical Approach", "Resection outcome (total, subtotal, partial)"]},

    {"target": "Non-intracranial infection",
     "features": preop_basic + ["Surgical Approach", "Resection outcome (total, subtotal, partial)"]},
]

# ---- Utility: safe compare + tune ----
def run_experiment(df, target, features, metric="AUC"):
    # Keep only columns that exist (avoid KeyError if some columns are missing)
    cols = [c for c in features + [target] if c in df.columns]
    if target not in cols:
        print(f"  [skip] Target column '{target}' not found.")
        return None, None

    # Drop rows where target is missing
    data = df[cols].copy()
    data = data[~data[target].isna()]
    if data[target].nunique() < 2:
        print(f"  [skip] Target '{target}' has <2 classes after dropping NaNs.")
        return None, None

    # PyCaret setup
    clf_setup = setup(
        data=data,
        target=target,
        session_id=42,
        fold=3,                      # safer with rare events
        fold_strategy="stratifiedkfold",
        fix_imbalance=True,          # handle class imbalance
        use_gpu=False,
        verbose=False
    )

    # Compare and enforce single estimator
    best_model = compare_models(sort=metric, n_select=1)
    # Guard: sometimes returns list
    if isinstance(best_model, list):
        best_model = best_model[0]

    if not hasattr(best_model, "fit"):
        print("  [skip] No valid estimator returned by compare_models.")
        return None, None

    # Tune
    try:
        tuned = tune_model(best_model, optimize=metric)
    except Exception as e:
        print(f"  [warn] tune_model failed for '{target}': {e}")
        tuned = best_model

    leaderboard = pull()
    return tuned, leaderboard

# ---- Run all experiments ----
all_leaderboards = {}
for exp in experiments:
    print(f"\n=== Running target: {exp['target']} ===")
    try:
        tuned, leaderboard = run_experiment(df_original, exp["target"], exp["features"], metric="AUC")
        if leaderboard is not None:
            all_leaderboards[exp["target"]] = leaderboard.copy()
            # Show the top few rows for quick inspection
            print(leaderboard.head())
    except Exception as e:
        print(f"  [error] {exp['target']}: {e}")

# Optionally, concatenate and save all results
if all_leaderboards:
    lb_all = pd.concat({k: v for k, v in all_leaderboards.items()}, names=["Target", "Row"]).reset_index(level=0)
    lb_all.to_csv("automl_leaderboards.csv", index=False)
    print("\nSaved: automl_leaderboards.csv")



=== Running target: Permanent DI ===


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.7491,0.7019,0.7685,0.8333,0.7990,0.4656,0.4700,0.9733
gbc,Gradient Boosting Classifier,0.6758,0.6991,0.8056,0.7222,0.7616,0.2533,0.2576,0.0733
et,Extra Trees Classifier,0.6996,0.6949,0.7685,0.7731,0.7691,0.3424,0.3455,0.1000
lda,Linear Discriminant Analysis,0.6245,0.6824,0.5741,0.7857,0.6627,0.2575,0.2759,0.0400
lr,Logistic Regression,0.7253,0.6759,0.7639,0.7937,0.7778,0.4112,0.4121,1.5500
xgboost,Extreme Gradient Boosting,0.6777,0.6676,0.7639,0.7465,0.7519,0.2771,0.2775,0.3767
nb,Naive Bayes,0.5018,0.6630,0.3472,0.7389,0.4683,0.1073,0.1316,1.0833
dt,Decision Tree Classifier,0.7271,0.6565,0.8796,0.7449,0.8040,0.3444,0.3704,1.0567
rf,Random Forest Classifier,0.6777,0.6102,0.8472,0.7121,0.7733,0.2303,0.2392,0.1033
catboost,CatBoost Classifier,0.7051,0.5676,0.8843,0.7264,0.7967,0.2726,0.2787,0.8533


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7143,0.6667,0.6667,0.8571,0.7500,0.4286,0.4472
1,0.6923,0.6667,0.6667,0.8571,0.7500,0.3659,0.3858
2,0.7692,0.7250,0.7500,0.8571,0.8000,0.5301,0.5367
Mean,0.7253,0.6861,0.6944,0.8571,0.7667,0.4415,0.4566
Std,0.0324,0.0275,0.0393,0.0000,0.0236,0.0677,0.0620


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.7143  0.6667  0.6667  0.8571  0.7500  0.4286  0.4472
1       0.6923  0.6667  0.6667  0.8571  0.7500  0.3659  0.3858
2       0.7692  0.7250  0.7500  0.8571  0.8000  0.5301  0.5367
Mean    0.7253  0.6861  0.6944  0.8571  0.7667  0.4415  0.4566
Std     0.0324  0.0275  0.0393  0.0000  0.0236  0.0677  0.0620

=== Running target: Other intracranial infection ===
  [error] Other intracranial infection: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 2, n_samples = 2

=== Running target: Non-intracranial infection ===
  [error] Non-intracranial infection: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fi

In [38]:
import pandas as pd
from pycaret.classification import setup, create_model, tune_model, finalize_model, save_model, predict_model, pull

# 1) Load raw data
df_original = pd.read_csv("data.csv")

# 2) Define features for Permanent DI (pre-op + intra-op you chose)
features_permdi = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Surgical Approach", "Resection outcome (total, subtotal, partial)"
]
target = "Permanent DI"

# 3) Keep only rows with target present
cols = [c for c in features_permdi + [target] if c in df_original.columns]
data = df_original[cols].dropna(subset=[target]).copy()

# 4) Pick a safe number of folds (avoid empty-class folds)
vc = data[target].value_counts()
minority = int(vc.min())
folds = max(2, min(3, minority))  # 2–3 folds depending on minority size

# 5) Setup (PyCaret will encode categoricals & impute internally)
_ = setup(
    data=data,
    target=target,
    session_id=42,
    fold=folds,
    fold_strategy="stratifiedkfold",
    fix_imbalance=True,    # helps with rare outcomes
    use_gpu=False,
    verbose=False
)

# 6) Train Ridge, then tune it (optimize F1 for class imbalance)
ridge = create_model("ridge")
ridge_tuned = tune_model(ridge, optimize="F1")

# (optional) Inspect CV leaderboard for tuned ridge
lb = pull()
print(lb.head())

# 7) Finalize on the full dataset (locks preprocessing + tuned params)
ridge_final = finalize_model(ridge_tuned)

# 8) Save the whole pipeline (preprocessing + model)
save_model(ridge_final, "permdi_ridge_v1")

# 9) (optional) Get CV/holdout predictions for sanity check
preds = predict_model(ridge_final)
print(preds.head())


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7857,0.6889,0.7778,0.8750,0.8235,0.5532,0.5594
1,0.7692,0.6667,0.7778,0.8750,0.8235,0.4935,0.5007
2,0.6923,0.7500,0.7500,0.7500,0.7500,0.3500,0.3500
Mean,0.7491,0.7019,0.7685,0.8333,0.7990,0.4656,0.4700
Std,0.0407,0.0352,0.0131,0.0589,0.0347,0.0853,0.0882


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7143,0.6667,0.6667,0.8571,0.7500,0.4286,0.4472
1,0.6923,0.6667,0.6667,0.8571,0.7500,0.3659,0.3858
2,0.7692,0.7250,0.7500,0.8571,0.8000,0.5301,0.5367
Mean,0.7253,0.6861,0.6944,0.8571,0.7667,0.4415,0.4566
Std,0.0324,0.0275,0.0393,0.0000,0.0236,0.0677,0.0620


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).
      Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold                                                          
0       0.7143  0.6667  0.6667  0.8571  0.7500  0.4286  0.4472
1       0.6923  0.6667  0.6667  0.8571  0.7500  0.3659  0.3858
2       0.7692  0.7250  0.7500  0.8571  0.8000  0.5301  0.5367
Mean    0.7253  0.6861  0.6944  0.8571  0.7667  0.4415  0.4566
Std     0.0324  0.0275  0.0393  0.0000  0.0236  0.0677  0.0620
Transformation Pipeline and Model Successfully Saved


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Ridge Classifier,0.8889,0.9167,0.8333,1.0000,0.9091,0.7692,0.7906


    Age         BMI  Tumor size  Tumor size.1  Tumor size.2  Tumor volume  \
32    65  26.040001         1.4           1.8           1.9      2.506989   
33    60  32.689999         4.1           3.0           2.0     12.880519   
14    64  28.620001         3.4           2.2           1.9      7.441380   
26    26  21.799999         3.7           4.1           2.8     22.240362   
44    73  21.549999         1.2           1.1           1.5      1.036725   

    Last BMI on file  \
32         29.100000   
33         33.900002   
14         29.549999   
26         21.600000   
44         20.600000   

    Pre-op visual impairment (y/n) - symptomatic complaint of visual loss  \
32                                                1.0                       
33                                                1.0                       
14                                                0.0                       
26                                                1.0                       
44     